In [ ]:

from py4stat import datania
import pandas as pd

# Generate 24 months of price data across markets
# We'll treat each market as a "business" and prices as "monthly turnover"
csv_file = datania.generate_price_data(n_months=24, start_year=2024, start_month=1, seed=42)
df = pd.read_csv(csv_file)

# Save to a file for DuckDB to query
df.to_csv('data/monthly_turnover.csv', index=False)
print(f"Created {len(df)} monthly records")
print(df.head())

In [ ]:
import duckdb

# Make sure you've run the dataset setup code first!

# Task 1: Basic query on a file
print("=== AVERAGE PRICE BY MARKET ===")

query1 = """
SELECT
    market,
    ROUND(AVG(price), 2) as avg_price,
    COUNT(*) as n_records
FROM 'data/monthly_turnover.csv'
GROUP BY market
ORDER BY avg_price DESC
"""

result1 = duckdb.query(query1).to_df()
print(result1)


# Task 2: Month-over-month changes with LAG
print("\n=== MONTH-OVER-MONTH PRICE CHANGES ===")

query2 = """
SELECT
    market,
    product_name,
    date,
    price,
    -- YOUR CODE HERE: Use LAG to get previous month's price
    -- LAG(price) OVER (PARTITION BY ??? ORDER BY ???) as prev_price,
    -- Calculate the change
    -- price - LAG(price) OVER (...) as price_change
FROM 'data/monthly_turnover.csv'
ORDER BY market, product_name, date
"""

# Uncomment and complete the query
# result2 = duckdb.query(query2).to_df()
# print(result2[result2['price_change'].notna()].nlargest(10, 'price_change'))


# Task 3: Rolling 3-month average and spike detection
print("\n=== ROLLING AVERAGE AND SPIKES ===")

query3 = """
SELECT
    market,
    product_name,
    date,
    price,
    -- YOUR CODE HERE: Calculate 3-month rolling average
    -- AVG(price) OVER (
    --     PARTITION BY market, product_name
    --     ORDER BY date
    --     ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    -- ) as rolling_avg_3m,
    -- Calculate spike ratio (current price / rolling average)
FROM 'data/monthly_turnover.csv'
ORDER BY market, product_name, date
"""

# Uncomment and complete
# result3 = duckdb.query(query3).to_df()
# print(result3.head(20))


# Task 4: Anomaly detection report
print("\n=== ANOMALY DETECTION REPORT ===")

query4 = """
WITH price_with_rolling AS (
    SELECT
        market,
        product_name,
        product_code,
        date,
        price,
        AVG(price) OVER (
            PARTITION BY market, product_code
            ORDER BY date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) as rolling_avg_3m
    FROM 'data/monthly_turnover.csv'
)
SELECT
    market,
    product_name,
    date,
    ROUND(price, 2) as current_price,
    ROUND(rolling_avg_3m, 2) as rolling_avg,
    ROUND(price / rolling_avg_3m, 2) as spike_ratio
FROM price_with_rolling
WHERE price > rolling_avg_3m * 1.5  -- Flag spikes > 1.5x average
  AND rolling_avg_3m IS NOT NULL  -- Exclude first 2 months (incomplete window)
ORDER BY spike_ratio DESC
LIMIT 20
"""

anomalies = duckdb.query(query4).to_df()
print(f"Found {len(anomalies)} price spikes (>1.5x rolling average):")
print(anomalies)

# Summary statistics
print(f"\n=== SUMMARY ===")
print(f"Total anomalies detected: {len(anomalies)}")
if len(anomalies) > 0:
    print(f"Highest spike ratio: {anomalies['spike_ratio'].max():.2f}x")
    print(f"Most affected market: {anomalies['market'].value_counts().idxmax()}")
    print(f"Most volatile product: {anomalies['product_name'].value_counts().idxmax()}")